# 05 - Action：迭代调整与延伸分析

> **STAR法则的第五步**：深入挖掘，处理业务干扰，迭代优化结论。

---

## 📋 学习目标

完成本notebook后，你将能够：
1. 使用CUPED降低方差，提高检验力
2. 执行后分层分析，识别效果异质性
3. 处理业务干扰（peeking、时间压力）
4. 进行敏感性分析

---

## 🚀 CUPED方差缩减

### 为什么需要CUPED？

```python
# 查看停留时长的方差
dwell_var = df_clean["dwell_time"].var()
historical_var = df_clean["historical_dwell_time"].var()

print(f"停留时长方差: {dwell_var:.2f}")
print(f"历史停留时长方差: {historical_var:.2f}")
print(f"\n方差大意味着：")
print(f"  - 检验力低（难以检测出真实效应）")
print(f"  - 需要更大样本量")
print(f"  - 置信区间宽")

# 计算变异系数
cv = df_clean["dwell_time"].std() / df_clean["dwell_time"].mean()
print(f"\n变异系数 (CV): {cv:.2f}")
print(f"(CV > 0.5 表示方差很大)")
```

### CUPED实现

**TODO 1**：实现CUPED调整。

```python
def cuped_adjustment(df, metric, covariate, group_col="group"):
    """
    CUPED (Controlled-experiment Using Pre-Experiment Data)
    利用实验前数据作为协变量，降低方差
    
    原理：
    Y_cuped = Y - θ * (X - mean(X))
    其中 θ = cov(Y, X) / var(X)
    
    参数:
        df: 数据框
        metric: 实验指标（如dwell_time）
        covariate: 协变量（如historical_dwell_time）
        group_col: 分组列名
    """
    df = df.copy()
    
    # TODO 1.1：计算协方差和方差
    # 提示：使用np.cov计算协方差矩阵
    
    # TODO 1.2：计算最优theta
    # θ = cov(Y, X) / var(X)
    
    # TODO 1.3：计算调整后的指标
    # Y_cuped = Y - θ * (X - mean(X))
    
    # 计算方差缩减比例
    original_var = df[metric].var()
    cuped_var = df["dwell_time_cuped"].var()
    variance_reduction = (original_var - cuped_var) / original_var
    
    print(f"CUPED调整结果：")
    print(f"  原始方差: {original_var:.2f}")
    print(f"  CUPED后方差: {cuped_var:.2f}")
    print(f"  方差缩减: {variance_reduction*100:.1f}%")
    print(f"  等价于样本量扩大: {1/(1-variance_reduction):.1f}倍")
    
    return df

# 应用CUPED
df_cuped = cuped_adjustment(df_clean, "dwell_time", "historical_dwell_time")
```

### CUPED后的检验

```python
# 用CUPED后的指标重新执行AB检验
print("="*60)
print("CUPED后的AB检验")
print("="*60)

result_cuped = ab_test(df_cuped, "dwell_time_cuped")

# 对比原始结果
print("\n" + "="*60)
print("对比：原始 vs CUPED")
print("="*60)

original_result = [r for r in results if r["metric"] == "dwell_time"][0]

print(f"{'指标':<15} {'原始':<12} {'CUPED':<12} {'改善':<12}")
print(f"{'p值':<15} {original_result['p_value']:<12.4f} {result_cuped['p_value']:<12.4f} {(original_result['p_value']/result_cuped['p_value']):<12.2f}x")
print(f"{'Cohen\'s d':<15} {original_result['cohens_d']:<12.4f} {result_cuped['cohens_d']:<12.4f} {(result_cuped['cohens_d']/original_result['cohens_d']):<12.2f}x")
print(f"{'CI宽度':<15} {(original_result['ci_upper']-original_result['ci_lower']):<12.2f} {(result_cuped['ci_upper']-result_cuped['ci_lower']):<12.2f}")
```

---

## 🎯 后分层分析

### 为什么需要后分层？

```python
print("后分层分析的目的：")
print("1. 识别不同用户群体的效果差异")
print("2. 发现潜在的机会或风险")
print("3. 为精细化运营提供依据")

print("\n例如：")
print("- 新用户 vs 老用户的效果是否不同？")
print("- iOS vs Android的效果是否不同？")
print("- 如果老用户效果显著但新用户不显著，上线策略应该如何调整？")
```

### 后分层实现

**TODO 2**：实现后分层分析函数。

```python
def stratified_analysis(df, metric, stratify_col, group_col="group", alpha=0.05):
    """
    后分层分析：按用户属性分层，看效果是否一致
    
    参数:
        df: 数据框
        metric: 分析指标
        stratify_col: 分层维度
        group_col: 分组列名
        alpha: 显著性水平
    """
    print(f"\n后分层分析 - 按 {stratify_col} 分层")
    print("="*60)
    
    # TODO 2.1：遍历每个分层
    for stratum in df[stratify_col].unique():
        if pd.isna(stratum):
            continue
        
        # TODO 2.2：提取该分层的数据
        
        # TODO 2.3：执行t检验
        
        # TODO 2.4：打印结果
        # 格式：分层值、样本量、效应值、p值、是否显著
        pass

# 按用户类型分层
stratified_analysis(df_clean, "dwell_time", "user_type")

# 按设备类型分层
stratified_analysis(df_clean, "dwell_time", "device_type")
```

### 可视化分层效果

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 按用户类型
user_type_data = []
for user_type in df_clean["user_type"].unique():
    subset = df_clean[df_clean["user_type"] == user_type]
    control = subset[subset["group"] == "control"]["dwell_time"]
    treatment = subset[subset["group"] == "treatment"]["dwell_time"]
    user_type_data.append([control, treatment])

axes[0].boxplot([d[0] for d in user_type_data], positions=[1, 4], widths=0.6)
axes[0].boxplot([d[1] for d in user_type_data], positions=[2, 5], widths=0.6)
axes[0].set_xticks([1.5, 4.5])
axes[0].set_xticklabels(["New", "Old"])
axes[0].set_title("Dwell Time by User Type")
axes[0].set_ylabel("Dwell Time (seconds)")
axes[0].legend(["Control", "Treatment"], loc="upper right")

# 按设备类型
device_type_data = []
for device_type in df_clean["device_type"].unique():
    subset = df_clean[df_clean["device_type"] == device_type]
    control = subset[subset["group"] == "control"]["dwell_time"]
    treatment = subset[subset["group"] == "treatment"]["dwell_time"]
    device_type_data.append([control, treatment])

axes[1].boxplot([d[0] for d in device_type_data], positions=[1, 4], widths=0.6)
axes[1].boxplot([d[1] for d in device_type_data], positions=[2, 5], widths=0.6)
axes[1].set_xticks([1.5, 4.5])
axes[1].set_xticklabels(["Android", "iOS"])
axes[1].set_title("Dwell Time by Device Type")
axes[1].set_ylabel("Dwell Time (seconds)")

plt.tight_layout()
plt.show()
```

---

## 🚨 业务干扰处理

### 干扰1：Peeking问题

```python
print("="*60)
print("业务干扰1：Peeking问题")
print("="*60)
print("\n场景：实验进行到一半（第7天），老板问：\")
print("'能不能提前看看结果？我下周要向董事会汇报。'\")
print("\n你的选择：")
print("A. 好的，我这就分析")
print("B. 不行，必须等实验结束")
print("C. 可以看，但要说明局限性")
```

**TODO 3**：模拟peeking的后果。

```python
def peeking_simulation(df, metric, n_peeks=5, alpha=0.05):
    """
    演示peeking问题：多次查看数据会提高假阳性率
    
    参数:
        df: 数据框
        metric: 查看的指标
        n_peeks: 查看次数
        alpha: 每次查看的显著性水平
    """
    print(f"\nPeeking问题演示：")
    print(f"  计划查看次数: {n_peeks}")
    print(f"  每次α={alpha}")
    print(f"  整体假阳性率 ≈ {1-(1-alpha)**n_peeks:.1%}")
    print(f"  (不是{alpha:.0%}，而是{1-(1-alpha)**n_peeks:.1%}！)")
    print()
    
    n_total = len(df)
    significant_peeks = 0
    
    for i in range(1, n_peeks + 1):
        n_sample = int(n_total * i / n_peeks)
        subset = df.iloc[:n_sample]
        
        control = subset[subset["group"] == "control"][metric].dropna()
        treatment = subset[subset["group"] == "treatment"][metric].dropna()
        
        if len(control) > 10 and len(treatment) > 10:
            t_stat, p_value = stats.ttest_ind(control, treatment, equal_var=False)
            is_significant = p_value < alpha
            if is_significant:
                significant_peeks += 1
            
            marker = " ***" if is_significant else ""
            print(f"  第{i}次查看 (n={n_sample}): p={p_value:.4f}{marker}")
    
    print(f"\n  显著次数: {significant_peeks}/{n_peeks}")
    print(f"  如果每次显著都停止实验，假阳性概率很高！")

# 执行peeking模拟
peeking_simulation(df_clean, "dwell_time")
```

### 干扰2：指标异常

```python
print("\n" + "="*60)
print("业务干扰2：指标异常")
print("="*60)
print("\n场景：实验组CTR显著为负（p<0.05），产品团队说：\")
print("'CTR下降可能是暂时的，再跑一周看看？'\")
print("\n你的思考：")

response = """
TODO：写下你的应对策略

1. CTR是guardrail指标，显著为负意味着什么？
2. "再跑一周"能解决问题吗？
3. 如果CTR持续为负，是否应该停止实验？
"""

print(response)
```

### 干扰3：时间压力

```python
print("\n" + "="*60)
print("业务干扰3：时间压力")
print("="*60)
print("\n场景：样本量计算显示需要2周，但产品团队说：\")
print("'我们等不了2周，1周必须出结论！'\")
print("\n你的选择：")

options = """
TODO：分析各选项的利弊

选项A：妥协，跑1周
  利：...
  弊：...

选项B：坚持2周
  利：...
  弊：...

选项C：折中，跑1周但提高α到0.1
  利：...
  弊：...
"""

print(options)
```

---

## 🔍 敏感性分析

**TODO 4**：执行敏感性分析。

```python
print("="*60)
print("敏感性分析")
print("="*60)

# 不同MDE下的样本量
mde_values = [0.10, 0.15, 0.20, 0.25]

print("\n不同MDE下的样本量需求：")
print(f"{'MDE':<10} {'每组样本量':<15} {'总样本量':<15} {'实验天数':<10}")
print("-"*60)

for mde in mde_values:
    n = sample_size_continuous(historical_mean, historical_std, mde)
    days = np.ceil(n * 2 / (dau * traffic_ratio))
    print(f"{mde*100:>6.0f}%    {n:<15,} {n*2:<15,} {days:<10.0f}")

# 不同α下的检验结果
print("\n不同α下的显著性：")
alpha_values = [0.01, 0.05, 0.10]

dwell_result = [r for r in results if r["metric"] == "dwell_time"][0]

for alpha_val in alpha_values:
    is_sig = dwell_result["p_value"] < alpha_val
    print(f"  α={alpha_val}: {'显著 ✓' if is_sig else '不显著 ✗'} (p={dwell_result['p_value']:.4f})")
```

---

## 📝 本节总结

完成本节，你应该已经：

- [ ] 使用CUPED降低方差
- [ ] 执行后分层分析
- [ ] 处理3个业务干扰场景
- [ ] 执行敏感性分析

**下一步**：进入 `06-result.ipynb`，总结结论并给出业务建议。

---

> 💡 **关键洞察**：
> 数据分析不是一次性工作，而是迭代过程。
> 第一次分析可能结论不对，调整后得到可靠结论。
